In [1]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   TS-FORECASTING KAGGLE — v42  (Per-horizon + N-BEATS h=1 + CS h=25)      ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  BASE: v41 (per-horizon feature selection, score 0.2605/Kaggle)            ║
║                                                                              ║
║  THAY ĐỔI SO VỚI v41:                                                      ║
║                                                                              ║
║  [1] 5 SEEDS thay vì 3                                                      ║
║      Bằng chứng: rerun.ipynb với 5 seeds cho score 0.2621 (tăng 0.0017)   ║
║      Runtime tăng ~67% nhưng variance giảm đáng kể                         ║
║                                                                              ║
║  [2] N-BEATS FEATURES — CHỈ CHO H=1                                        ║
║      Lý thuyết (arXiv:1905.10437): phân tách trend + seasonality           ║
║      Bằng chứng: rerun.ipynb N-BEATS giúp h=1 +0.0102 (rất mạnh)         ║
║      Features:                                                               ║
║        tslope  : (x_t - x_{t-9}) / 9  ← linear trend slope                ║
║        taccel  : diff1_t - diff1_{t-5} ← trend curvature                  ║
║        detrend10: x - rolling_mean(10) ← short-cycle residual              ║
║        detrend25: x - rolling_mean(25) ← medium-cycle residual             ║
║      Source: feature_cg, feature_al (top-2 h=1) → 8 features mới         ║
║      Tại sao CHỈ h=1: rerun gây h=3 -0.017, h=10 -0.012 khi dùng cho all ║
║                                                                              ║
║  [3] CROSS-SECTIONAL FEATURES — CHỈ CHO H=25                              ║
║      Lý thuyết: h=25 cần WHERE series is relative to peers (cross-section) ║
║      không cần WHERE was relative to itself (time-series lag)              ║
║      Bằng chứng: feature_v raw #2, feature_bg raw #2, feature_l raw #4    ║
║        → raw features dominante, KHÔNG có lag/temporal ở top 20 h=25      ║
║      CS features: z-score per ts_index (NOT temporal, SAFE for h=25)      ║
║        feature_v_cs, feature_bg_cs, feature_l_cs → 3 features             ║
║                                                                              ║
║  PER-HORIZON FEATURE MAP (GIỮ NGUYÊN V41):                                ║
║    h=1:  v38(193) + surgical(7) + nbeats(8)                               ║
║    h=3:  v38(193) + cg_ema_cross(1)                                        ║
║    h=10: v38(193) — pure v38                                               ║
║    h=25: v38(193) + cs_long(3)                                             ║
║                                                                              ║
║  3 VARIANTS ĐỂ CHẠY SONG SONG (xem CONFIG bên dưới):                      ║
║    v42  : main (đây là file này)                                           ║
║    v42b : VARIANT_ID='b' → lambda_l2=15.0 cho h=25                        ║
║    v42c : VARIANT_ID='c' → num_leaves=127 cho h=10/h=25                   ║
║    v42d : VARIANT_ID='d' → feature_fraction=0.55 cho h=25                 ║
║  → Sao chép notebook, đổi VARIANT_ID duy nhất                             ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import gc
import os
import time
import warnings
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import lightgbm as lgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
T0 = time.time()

# ══════════════════════════════════════════════════════════════════════════════
# ★ VARIANT CONFIG — ĐỔI DÒNG NÀY KHI SAO CHÉP NOTEBOOK ★
# ══════════════════════════════════════════════════════════════════════════════
VARIANT_ID = 'a'   # 'a'=main | 'b'=lambda_l2=15 h25 | 'c'=leaves=127 h10/25 | 'd'=ff=0.55 h25

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
TRAIN_PATH    = '/kaggle/input/competitions/ts-forecasting/train.parquet'
TEST_PATH     = '/kaggle/input/competitions/ts-forecasting/test.parquet'
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH, TEST_PATH = 'train.parquet', 'test.parquet'

VAL_THRESHOLD = 3500
HORIZONS      = [1, 3, 10, 25]
SEEDS         = [42, 2024, 12345, 777, 9999]   # 5 seeds (rerun insight)
CLIP_Q_LOW    = 0.005
CLIP_Q_HIGH   = 0.995

# Base LGB params (giữ nguyên v38)
LGB_BASE = {
    'objective':         'regression',
    'metric':            'rmse',
    'learning_rate':     0.015,
    'n_estimators':      5000,
    'num_leaves':        127,
    'min_child_samples': 200,
    'feature_fraction':  0.65,
    'bagging_fraction':  0.75,
    'bagging_freq':      5,
    'lambda_l1':         0.1,
    'lambda_l2':        10.0,
    'verbosity':         -1,
    'n_jobs':            -1,
}

# Per-horizon LGB overrides (variant-specific)
def get_lgb_params(horizon):
    """
    Trả về LGB params cho từng horizon.
    VARIANT_ID thay đổi params cho h=10/h=25.
    """
    p = dict(LGB_BASE)
    if VARIANT_ID == 'b' and horizon == 25:
        # Variant b: stronger L2 regularization cho h=25
        # Giả thuyết: CS features mới có thể overfit → regularize mạnh hơn
        p['lambda_l2'] = 15.0
        print(f'      [variant b] h=25: lambda_l2=15.0')
    elif VARIANT_ID == 'c' and horizon in [10, 25]:
        # Variant c: num_leaves=127 cho h=10/h=25
        # Giả thuyết: long-horizon cần complex decision boundaries hơn
        p['num_leaves'] = 127
        print(f'      [variant c] h={horizon}: num_leaves=127')
    elif VARIANT_ID == 'd' and horizon == 25:
        # Variant d: feature_fraction=0.55 cho h=25
        # Giả thuyết: CS features mới + higher randomness → better generalization
        p['feature_fraction'] = 0.55
        print(f'      [variant d] h=25: feature_fraction=0.55')
    return p

# ── Per-horizon feature exclusion ─────────────────────────────────────────────
# Tất cả features được TẠO trong FE, nhưng mỗi horizon DÙNG subset khác nhau
SURGICAL_EMA = ['feature_cg_ema_cross', 'feature_v_ema_cross']
SURGICAL_V_T = ['feature_v_roll_std_20', 'feature_v_lag5', 'feature_v_lag25', 'feature_v_roc10']
ALL_SURGICAL  = SURGICAL_EMA + SURGICAL_V_T + ['feature_v_rank']  # 7 total
NBEATS_FEATS  = [f'{c}_{s}' for c in ['feature_cg', 'feature_al']
                             for s in ['tslope','taccel','detrend10','detrend25']]
CS_LONG_FEATS = ['feature_v_cs', 'feature_bg_cs', 'feature_l_cs']

HORIZON_EXCLUDE = {
    # h=1: giữ tất cả (v38 + 7 surgical + 8 N-BEATS); loại CS_LONG
    1:  CS_LONG_FEATS,

    # h=3: chỉ cg_ema_cross; loại 6 surgical + N-BEATS + CS_LONG
    3:  [f for f in ALL_SURGICAL if f != 'feature_cg_ema_cross']
        + NBEATS_FEATS + CS_LONG_FEATS,

    # h=10: pure v38; loại tất cả new features
    10: ALL_SURGICAL + NBEATS_FEATS + CS_LONG_FEATS,

    # h=25: v38 + CS_LONG (3 CS features); loại surgical + N-BEATS
    25: ALL_SURGICAL + NBEATS_FEATS,
}

print(f'\n>>> v42 variant={VARIANT_ID} | seeds={len(SEEDS)} | VAL_THRESHOLD={VAL_THRESHOLD}')
print(f'>>> Feature map:')
for h, excl in HORIZON_EXCLUDE.items():
    n_base = 193; n_excl = len([f for f in excl])
    print(f'    h={h}: ~{n_base + 7 + 8 + 3 - n_excl} feats '
          f'(exclude {n_excl}: {[f.replace("feature_","").replace("_cs","_CS") for f in excl[:3]]}...)')

# ══════════════════════════════════════════════════════════════════════════════
# UTILITIES
# ══════════════════════════════════════════════════════════════════════════════
def _clip01(x): return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w):
    y_target = np.asarray(y_target, dtype=np.float64)
    y_pred   = np.asarray(y_pred,   dtype=np.float64)
    w        = np.asarray(w,        dtype=np.float64)
    denom    = np.sum(w * y_target**2)
    if denom <= 0: return 0.0
    return float(np.sqrt(1.0 - _clip01(np.sum(w*(y_target-y_pred)**2) / denom)))

def log_ram(label=''):
    try:
        import psutil
        mb = psutil.Process(os.getpid()).memory_info().rss / 1024**2
        print(f'   [RAM {label}]: {mb:,.0f} MB ({mb/1024:.1f} GB)')
    except ImportError:
        pass

# ══════════════════════════════════════════════════════════════════════════════
# FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════════════════
def build_features_polars(train_path, test_path):
    print(f'⏳ FE v42 (variant={VARIANT_ID} | N-BEATS h=1 | CS h=25 | 5 seeds)...')

    # ── Load & concat ─────────────────────────────────────────────────────────
    df_train = pl.read_parquet(train_path)
    df_test  = pl.read_parquet(test_path)
    df_train = df_train.with_columns(pl.lit(1, dtype=pl.Int8).alias('is_train'))
    df_test  = df_test.with_columns(
        pl.lit(0, dtype=pl.Int8).alias('is_train'),
        pl.lit(None).cast(pl.Float64).alias('y_target'),
        pl.lit(None).cast(pl.Float64).alias('weight'),
    ).select(df_train.columns)
    df = pl.concat([df_train, df_test]).sort(
        ['code', 'sub_code', 'sub_category', 'horizon', 'ts_index'])
    del df_train, df_test; gc.collect()
    print(f'   concat: {len(df):,} rows | {df.width} cols')
    log_ram('after concat')

    # ── Target Encoding — fit CHỈ trên train ≤ 3500 ──────────────────────────
    fit_df = df.filter((pl.col('is_train')==1) & (pl.col('ts_index')<=VAL_THRESHOLD))
    gm     = float(fit_df.select(pl.col('y_target').mean()).item())
    cat_e  = fit_df.group_by('sub_category').agg(pl.col('y_target').mean().alias('sub_category_enc'))
    cod_e  = fit_df.group_by('sub_code').agg(pl.col('y_target').mean().alias('sub_code_enc'))
    df = df.join(cat_e, on='sub_category', how='left').with_columns(pl.col('sub_category_enc').fill_null(gm))
    df = df.join(cod_e, on='sub_code',     how='left').with_columns(pl.col('sub_code_enc').fill_null(gm))
    del fit_df, cat_e, cod_e; gc.collect()

    # ── Interactions ──────────────────────────────────────────────────────────
    exprs = []
    if 'feature_al' in df.columns and 'feature_am' in df.columns:
        exprs += [(pl.col('feature_al')-pl.col('feature_am')).alias('d_al_am'),
                  (pl.col('feature_al')/(pl.col('feature_am')+1e-7)).alias('r_al_am')]
    if 'feature_cg' in df.columns and 'feature_by' in df.columns:
        exprs.append((pl.col('feature_cg')-pl.col('feature_by')).alias('d_cg_by'))
    if exprs:
        df = df.with_columns(exprs); exprs.clear()

    # ── CS z-score (v36 original cols) ───────────────────────────────────────
    for c in ['feature_al','feature_am','feature_cg','feature_by','d_al_am']:
        if c in df.columns:
            exprs.append(
                ((pl.col(c)-pl.col(c).mean().over('ts_index'))/(pl.col(c).std().over('ts_index')+1e-7))
                .alias(f'{c}_cs'))
    exprs += [(np.sin(2*np.pi*pl.col('ts_index')/100.0)).alias('t_sin'),
              (np.cos(2*np.pi*pl.col('ts_index')/100.0)).alias('t_cos')]
    df = df.with_columns(exprs); exprs.clear()

    # ── [3] CS features for h=25 (safe: no temporal autocorr) ────────────────
    # Tính z-score cross-sectional cho 3 cols quan trọng nhất của h=25
    # Lý do an toàn: computed at ts_index level, NOT per time-series
    for c in ['feature_v', 'feature_bg', 'feature_l']:
        if c in df.columns:
            exprs.append(
                ((pl.col(c)-pl.col(c).mean().over('ts_index'))/(pl.col(c).std().over('ts_index')+1e-7))
                .cast(pl.Float32).alias(f'{c}_cs'))
    if exprs:
        df = df.with_columns(exprs); exprs.clear()
    print(f'   ✓ CS_LONG features: feature_v_cs, feature_bg_cs, feature_l_cs')

    # ── v38 Lag + Rolling + DiffRoC ───────────────────────────────────────────
    LAG_COLS = ['feature_al','feature_am','feature_cg','feature_by','feature_s']
    src = [c for c in LAG_COLS if c in df.columns]
    grp = ['code','sub_code','sub_category','horizon']

    for c in src:
        for lag in [1,3,5,10,25]:
            exprs.append(pl.col(c).shift(lag).over(grp).alias(f'{c}_lag{lag}'))
        for w in [5,10,20]:
            exprs.append(pl.col(c).rolling_mean(w,min_periods=1).over(grp).alias(f'{c}_roll_mean_{w}'))
            exprs.append(pl.col(c).rolling_std(w,min_periods=1).over(grp).alias(f'{c}_roll_std_{w}'))
        exprs.append(pl.col(c).ewm_mean(span=10,min_periods=1,ignore_nulls=True).over(grp).alias(f'{c}_ewm10'))
        exprs.append(pl.col(c).diff(1).over(grp).alias(f'{c}_diff1'))
        exprs.append((pl.col(c).rank()/pl.count()).over('ts_index').alias(f'{c}_rank'))
        # DiffRoC
        exprs.append(pl.col(c).diff(3).over(grp).cast(pl.Float32).alias(f'{c}_diff3'))
        exprs.append(pl.col(c).diff(5).over(grp).cast(pl.Float32).alias(f'{c}_diff5'))
        for k in [5,10]:
            sh = pl.col(c).shift(k).over(grp)
            exprs.append(((pl.col(c)-sh)/(sh.abs()+1e-7)).clip(-5.,5.).cast(pl.Float32).alias(f'{c}_roc{k}'))
        exprs.append((pl.col(c)-2*pl.col(c).shift(1).over(grp)+pl.col(c).shift(2).over(grp))
                     .cast(pl.Float32).alias(f'{c}_accel'))
    df = df.with_columns(exprs); exprs.clear()
    print(f'   ✓ v38 lag/rolling/DiffRoC: {len(src)} source cols')

    # ── [v41] Surgical features ───────────────────────────────────────────────
    for col in ['feature_cg','feature_v']:
        if col not in df.columns: continue
        df = df.with_columns([
            pl.col(col).ewm_mean(span=5,min_periods=1,ignore_nulls=True).over(grp).cast(pl.Float32).alias('__e5'),
            pl.col(col).ewm_mean(span=25,min_periods=1,ignore_nulls=True).over(grp).cast(pl.Float32).alias('__e25'),
        ])
        df = df.with_columns((pl.col('__e5')-pl.col('__e25')).cast(pl.Float32).alias(f'{col}_ema_cross'))
        df = df.drop(['__e5','__e25']); gc.collect()

    if 'feature_v' in df.columns:
        c = 'feature_v'
        df = df.with_columns([
            (pl.col(c).rank()/pl.count()).over('ts_index').cast(pl.Float32).alias('feature_v_rank'),
            pl.col(c).rolling_std(20,min_periods=2).over(grp).cast(pl.Float32).alias('feature_v_roll_std_20'),
            pl.col(c).shift(5).over(grp).cast(pl.Float32).alias('feature_v_lag5'),
            pl.col(c).shift(25).over(grp).cast(pl.Float32).alias('feature_v_lag25'),
            ((pl.col(c)-pl.col(c).shift(10).over(grp))/(pl.col(c).shift(10).over(grp).abs()+1e-7))
            .clip(-5.,5.).cast(pl.Float32).alias('feature_v_roc10'),
        ])

    # ── [2] N-BEATS features — tạo cho tất cả, filter khi train (h=1 only) ──
    # Inspired by N-BEATS (arXiv:1905.10437): decompose series into trend + seasonality
    # Trend stack:   polynomial basis approximation
    # Season stack:  Fourier basis residual (= detrended)
    # Simplified implementation using rolling ops (no actual basis expansion)
    print('   [v42] Computing N-BEATS decomposition features...')
    NBEATS_SRC = ['feature_cg', 'feature_al']  # top-2 h=1 source cols
    for c in NBEATS_SRC:
        if c not in df.columns: continue
        # Trend slope: (x_t - x_{t-9}) / 9 ≈ local linear trend gradient
        exprs.append(
            ((pl.col(c) - pl.col(c).shift(9).over(grp)) / 9.0)
            .cast(pl.Float32).alias(f'{c}_tslope'))
        # Trend acceleration: d(diff1)/dt — how trend is changing
        exprs.append(
            (pl.col(c).diff(1).over(grp) - pl.col(c).diff(1).shift(5).over(grp))
            .cast(pl.Float32).alias(f'{c}_taccel'))
        # Detrend10: seasonality residual over 10-period window
        # = current value - slow trend → captures short cycles
        exprs.append(
            (pl.col(c) - pl.col(c).rolling_mean(10, min_periods=1).over(grp))
            .cast(pl.Float32).alias(f'{c}_detrend10'))
        # Detrend25: residual over 25-period → medium-cycle seasonality
        exprs.append(
            (pl.col(c) - pl.col(c).rolling_mean(25, min_periods=1).over(grp))
            .cast(pl.Float32).alias(f'{c}_detrend25'))
    if exprs:
        df = df.with_columns(exprs); exprs.clear()
    print(f'   ✓ N-BEATS features: {len(NBEATS_SRC)*4} cols (will be filtered to h=1 only)')

    # ── Final cast & fill ─────────────────────────────────────────────────────
    df = df.with_columns(cs.float().cast(pl.Float32).fill_null(0.0))
    total = df.width
    print(f'   Total FE cols: {total}')
    log_ram('before to_pandas')

    df_tr = df.filter(pl.col('is_train')==1).drop('is_train').to_pandas()
    df_te = df.filter(pl.col('is_train')==0).drop(['is_train','y_target','weight']).to_pandas()
    del df; gc.collect()
    log_ram('FE done')
    return df_tr, df_te


# ══════════════════════════════════════════════════════════════════════════════
# TRAIN PER-HORIZON
# ══════════════════════════════════════════════════════════════════════════════
def solve_horizon(horizon, train_df, test_df):
    t_h = time.time()
    print(f"\n{'='*70}\n 🚀 HORIZON {horizon}  [variant={VARIANT_ID}]\n{'='*70}")

    tr = train_df[train_df['horizon']==horizon].copy()
    te = test_df[test_df['horizon']==horizon].copy()

    EXCL = {'id','code','sub_code','sub_category','horizon','ts_index','weight','y_target'}
    excl_h = set(HORIZON_EXCLUDE.get(horizon, []))
    feats  = [c for c in tr.columns if c not in EXCL and c not in excl_h
              and c in te.columns]  # safety: only use cols that exist in test too

    # Log which feature groups are active
    n_surg  = sum(1 for c in feats if c in set(ALL_SURGICAL))
    n_nbeat = sum(1 for c in feats if c in set(NBEATS_FEATS))
    n_cs    = sum(1 for c in feats if c in set(CS_LONG_FEATS))
    print(f'Data: train={len(tr):,}, test={len(te):,} | '
          f'Feats={len(feats)} [surgical={n_surg} nbeats={n_nbeat} cs={n_cs}]')

    fit_m = tr['ts_index'] <= VAL_THRESHOLD
    val_m = tr['ts_index'] >  VAL_THRESHOLD

    X_fit = tr.loc[fit_m, feats]; y_fit = tr.loc[fit_m,'y_target']; w_fit = tr.loc[fit_m,'weight']
    X_val = tr.loc[val_m, feats]; y_val = tr.loc[val_m,'y_target']; w_val = tr.loc[val_m,'weight']
    X_all = tr[feats];            y_all = tr['y_target'];           w_all = tr['weight']
    ts_val = tr.loc[val_m,'ts_index'].values

    val_pred  = np.zeros(len(y_val), dtype=np.float64)
    test_pred = np.zeros(len(te),    dtype=np.float64)
    best_iters = []
    fi_cum    = np.zeros(len(feats), dtype=np.float64)

    lgb_params = get_lgb_params(horizon)

    for i, seed in enumerate(SEEDS, 1):
        print(f'   -> Seed {i}/{len(SEEDS)} (seed={seed})...')

        mdl_val = lgb.LGBMRegressor(**lgb_params, random_state=seed)
        mdl_val.fit(X_fit, y_fit, sample_weight=w_fit,
                    eval_set=[(X_val, y_val)], eval_sample_weight=[w_val],
                    callbacks=[lgb.early_stopping(200, verbose=False),
                                lgb.log_evaluation(period=-1)])
        bi = max(int(mdl_val.best_iteration_ or lgb_params['n_estimators']), 20)
        best_iters.append(bi)
        val_pred += mdl_val.predict(X_val) / len(SEEDS)
        del mdl_val; gc.collect()

        mdl_full = lgb.LGBMRegressor(**{**lgb_params, 'n_estimators': bi}, random_state=seed)
        mdl_full.fit(X_all, y_all, sample_weight=w_all,
                     callbacks=[lgb.log_evaluation(period=-1)])
        test_pred += mdl_full.predict(te[feats]) / len(SEEDS)
        fi_cum    += mdl_full.feature_importances_
        del mdl_full; gc.collect()

    score     = weighted_rmse_score(y_val.values, val_pred, w_val.values)
    q_lo, q_hi = np.quantile(y_fit.values, [CLIP_Q_LOW, CLIP_Q_HIGH])
    test_clip = np.clip(test_pred, q_lo, q_hi)

    print(f'💡 h={horizon} | score={score:.6f} | avg_iter={np.mean(best_iters):.0f} | '
          f'elapsed={(time.time()-t_h)/60:.1f} min')

    fi_df = pd.DataFrame({'feature': feats, 'importance': fi_cum/len(SEEDS)})\
              .sort_values('importance', ascending=False).reset_index(drop=True)

    out = {'horizon': horizon, 'id_test': te['id'].values,
           'test_pred_raw': test_pred, 'test_pred_clip': test_clip,
           'id_val': tr.loc[val_m,'id'].values, 'y_val': y_val.values,
           'w_val': w_val.values, 'val_pred': val_pred,
           'ts_val': ts_val, 'score_local': score,
           'best_iters': best_iters, 'feature_imp': fi_df, 'n_feats': len(feats)}

    del tr, te, X_fit, y_fit, w_fit, X_val, y_val, w_val, X_all, y_all, w_all
    del val_pred, test_pred, test_clip, fi_cum, ts_val
    gc.collect()
    return out


# ══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC PLOTS
# ══════════════════════════════════════════════════════════════════════════════
def create_diagnostic_plots(all_outputs):
    print('\n📊 Generating diagnostic plots...')
    HC   = {1:'#2196F3',3:'#4CAF50',10:'#FF9800',25:'#E91E63'}
    V41  = {1:0.0766, 3:0.1235, 10:0.2282, 25:0.2730}
    DARK = '#111827'

    # Plot 1: Score comparison
    fig, ax = plt.subplots(figsize=(9, 4.5))
    fig.patch.set_facecolor(DARK); ax.set_facecolor('#1f2937')
    h_list = sorted([o['horizon'] for o in all_outputs])
    s42 = [next(o['score_local'] for o in all_outputs if o['horizon']==h) for h in h_list]
    s41 = [V41[h] for h in h_list]
    x = np.arange(len(h_list))
    ax.bar(x-0.22, s41, 0.38, label=f'v41 (Kaggle≈0.2605)', color='#4B5563', edgecolor='#6B7280')
    ax.bar(x+0.22, s42, 0.38, label=f'v42-{VARIANT_ID}', color=[HC[h] for h in h_list], edgecolor='#9CA3AF')
    for i,(so,sn) in enumerate(zip(s41,s42)):
        d=sn-so; sign='+' if d>=0 else ''
        ax.text(i+0.22, sn+0.003, f'{sign}{d:.4f}', ha='center', va='bottom',
                fontsize=9, color='#4ade80' if d>=0 else '#f87171')
    ax.set_xticks(x)
    ax.set_xticklabels([f'h={h}\n({o["n_feats"]} feats)' for h,o in
                        zip(h_list, sorted(all_outputs, key=lambda o:o['horizon']))], color='#D1D5DB')
    ax.tick_params(colors='#9CA3AF'); ax.set_ylabel('Local Score', color='#D1D5DB')
    ax.set_title(f'v41 vs v42-{VARIANT_ID}\nh=1: +N-BEATS | h=25: +CS | 5 seeds', color='#F9FAFB', pad=10)
    ax.legend(fontsize=9, facecolor='#374151', labelcolor='#D1D5DB', edgecolor='#4B5563')
    for sp in ax.spines.values(): sp.set_edgecolor('#374151')
    plt.tight_layout()
    plt.savefig('diag1_scores.png', dpi=130, bbox_inches='tight', facecolor=DARK)
    plt.close(); print('   ✓ diag1_scores.png')

    # Plot 2: OOF scatter
    fig, axes = plt.subplots(1,4, figsize=(18,4.5))
    fig.patch.set_facecolor(DARK)
    for ax, out in zip(axes, sorted(all_outputs, key=lambda o:o['horizon'])):
        h=out['horizon']; ax.set_facecolor('#1f2937')
        yt=out['y_val']; yp=out['val_pred']
        idx=np.random.choice(len(yt), min(3000,len(yt)), replace=False)
        ax.scatter(yt[idx],yp[idx],s=4,alpha=0.3,color=HC[h])
        lo=min(yt.min(),yp.min()); hi=max(yt.max(),yp.max())
        ax.plot([lo,hi],[lo,hi],'w--',lw=1,alpha=0.5)
        ax.set_title(f'h={h} score={out["score_local"]:.4f} (v41:{V41[h]:.4f})',
                     color='#F9FAFB', fontsize=9)
        ax.set_xlabel('y_true',color='#9CA3AF',fontsize=8); ax.set_ylabel('y_pred',color='#9CA3AF',fontsize=8)
        ax.tick_params(colors='#6B7280',labelsize=7)
        for sp in ax.spines.values(): sp.set_edgecolor('#374151')
    plt.suptitle('OOF Pred vs True',color='#D1D5DB',fontsize=11,y=1.01)
    plt.tight_layout()
    plt.savefig('diag2_pred_vs_true.png', dpi=130, bbox_inches='tight', facecolor=DARK)
    plt.close(); print('   ✓ diag2_pred_vs_true.png')

    # Plot 3: Temporal residual
    fig, axes = plt.subplots(1,4, figsize=(18,4))
    fig.patch.set_facecolor(DARK)
    for ax, out in zip(axes, sorted(all_outputs, key=lambda o:o['horizon'])):
        h=out['horizon']; ax.set_facecolor('#1f2937')
        ts=out['ts_val']; res=out['y_val']-out['val_pred']
        bins=np.percentile(ts,np.linspace(0,100,51))
        bx,by,be=[],[],[]
        for j in range(len(bins)-1):
            m=(ts>=bins[j])&(ts<bins[j+1])
            if m.sum()>20:
                bx.append((bins[j]+bins[j+1])/2); by.append(np.mean(res[m]))
                be.append(np.std(res[m])/np.sqrt(m.sum()))
        bx,by,be=map(np.array,[bx,by,be])
        ax.plot(bx,by,color=HC[h],lw=1.5)
        ax.fill_between(bx,by-be,by+be,alpha=0.2,color=HC[h])
        ax.axhline(0,color='white',ls='--',lw=0.8,alpha=0.5)
        ax.set_title(f'h={h} | {out["n_feats"]} feats', color='#F9FAFB', fontsize=9)
        ax.set_xlabel('ts_index',color='#9CA3AF',fontsize=8)
        ax.set_ylabel('mean residual',color='#9CA3AF',fontsize=8)
        ax.tick_params(colors='#6B7280',labelsize=7)
        for sp in ax.spines.values(): sp.set_edgecolor('#374151')
    plt.suptitle('Temporal residual — Phẳng=tốt | Spike=regime shift',
                 color='#D1D5DB',fontsize=11,y=1.01)
    plt.tight_layout()
    plt.savefig('diag3_temporal_residual.png', dpi=130, bbox_inches='tight', facecolor=DARK)
    plt.close(); print('   ✓ diag3_temporal_residual.png')

    # Plot 4: Feature importance
    SURG_SET  = set(ALL_SURGICAL)
    NBEAT_SET = set(NBEATS_FEATS)
    CS_SET    = set(CS_LONG_FEATS)
    DIFF_KEYS = ['_diff3','_diff5','_roc5','_roc10','_accel']

    fig, axes = plt.subplots(2,2, figsize=(18,14))
    fig.patch.set_facecolor(DARK)
    from matplotlib.patches import Patch
    for ax, out in zip(axes.flatten(), sorted(all_outputs, key=lambda o:o['horizon'])):
        h=out['horizon']; fi=out['feature_imp'].head(20)
        ax.set_facecolor('#1f2937')
        colors=[]
        for feat in fi['feature']:
            if feat in NBEAT_SET:   colors.append('#10b981')  # green = N-BEATS
            elif feat in CS_SET:    colors.append('#8b5cf6')  # purple = CS_LONG
            elif feat in SURG_SET:  colors.append('#f59e0b')  # amber = surgical
            elif any(x in feat for x in DIFF_KEYS): colors.append('#22d3ee')  # cyan = DiffRoC
            else: colors.append(HC[h])
        ax.barh(range(len(fi)), fi['importance'], color=colors, edgecolor='#374151')
        ax.set_yticks(range(len(fi))); ax.set_yticklabels(fi['feature'],fontsize=7,color='#D1D5DB')
        ax.invert_yaxis()
        ax.set_title(f'h={h} | {out["n_feats"]} feats used', color='#F9FAFB', fontsize=10)
        ax.set_xlabel('importance',color='#9CA3AF',fontsize=8)
        ax.tick_params(colors='#6B7280',labelsize=7)
        for sp in ax.spines.values(): sp.set_edgecolor('#374151')
        leg=[Patch(facecolor='#10b981',label='N-BEATS (h=1 only)'),
             Patch(facecolor='#8b5cf6',label='CS_LONG (h=25 only)'),
             Patch(facecolor='#f59e0b',label='Surgical (h=1/h=3)'),
             Patch(facecolor='#22d3ee',label='DiffRoC'),
             Patch(facecolor=HC[h],   label='v36/base')]
        ax.legend(handles=leg, fontsize=7, facecolor='#374151', labelcolor='#D1D5DB',
                  edgecolor='#4B5563', loc='lower right')
    plt.suptitle(f'Feature Importance v42-{VARIANT_ID}\n'
                 'Xanh=N-BEATS | Tím=CS_LONG | Vàng=Surgical | Cyan=DiffRoC | Màu=v36',
                 color='#D1D5DB',fontsize=11,y=1.01)
    plt.tight_layout()
    plt.savefig('diag4_feature_importance.png', dpi=130, bbox_inches='tight', facecolor=DARK)
    plt.close(); print('   ✓ diag4_feature_importance.png')


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == '__main__':

    df_tr_all, df_te_all = build_features_polars(TRAIN_PATH, TEST_PATH)
    log_ram('after FE')

    all_outputs = []
    for h in HORIZONS:
        all_outputs.append(solve_horizon(h, df_tr_all, df_te_all))

    sub_raw_p, sub_clip_p, oof_p = [], [], []
    for out in all_outputs:
        sub_raw_p.append(pd.DataFrame({'id':out['id_test'],'prediction':out['test_pred_raw']}))
        sub_clip_p.append(pd.DataFrame({'id':out['id_test'],'prediction':out['test_pred_clip']}))
        oof_p.append(pd.DataFrame({'id':out['id_val'],'y_true':out['y_val'],
                                   'y_pred':out['val_pred'],'w':out['w_val'],'horizon':out['horizon']}))

    sub_raw  = pd.concat(sub_raw_p,  ignore_index=True)
    sub_clip = pd.concat(sub_clip_p, ignore_index=True)
    oof      = pd.concat(oof_p,      ignore_index=True)
    del sub_raw_p, sub_clip_p, oof_p; gc.collect()

    test_order = pd.read_parquet(TEST_PATH, columns=['id'])
    sub_clip = test_order.merge(sub_clip, on='id', how='left').fillna(0)
    sub_raw  = test_order.merge(sub_raw,  on='id', how='left').fillna(0)
    del test_order

    agg = weighted_rmse_score(oof['y_true'].values, oof['y_pred'].values, oof['w'].values)

    create_diagnostic_plots(all_outputs)

    fname_suf = f'_v42{VARIANT_ID}'
    sub_clip.to_csv('submission.csv',                  index=False)
    sub_raw.to_csv( f'submission_noclip{fname_suf}.csv', index=False)
    oof.to_csv(     f'oof{fname_suf}.csv',               index=False)

    V41_REF = {1:0.0766, 3:0.1235, 10:0.2282, 25:0.2730}
    V38_REF = {1:0.0725, 3:0.1420, 10:0.2282, 25:0.2730}
    print(f"\n{'='*70}")
    print(f'🏆 v42-{VARIANT_ID}  ({len(SEEDS)} seeds | N-BEATS h=1 | CS h=25)')
    print(f'   ref: v38 Kaggle=0.2604 | v41 Kaggle≈0.2605')
    print(f"{'─'*70}")
    print(f'  {"h":>4} | {"feats":>6} | {"v41":>8} | {"v42":>8} | {"vs_v38":>8} | iter')
    for out in sorted(all_outputs, key=lambda o:o['horizon']):
        h=out['horizon']; s=out['score_local']
        d41=s-V41_REF[h]; d38=s-V38_REF[h]
        t41='↑' if d41>=0 else '↓'
        t38='↑' if d38>=0 else '↓'
        print(f'  h={h:>2} | {out["n_feats"]:>6} | {V41_REF[h]:>8.4f} | {s:>8.4f} | {t38}{abs(d38):.4f} | {np.mean(out["best_iters"]):.0f}')
    print(f"{'─'*70}")
    print(f'🔥 Aggregate Local Score: {agg:.6f}')
    print(f"⏱️  Tổng thời gian: {(time.time()-T0)/60:.1f} phút")
    print(f"{'='*70}")

    print(f'''
CHECKLIST v42-{VARIANT_ID}:

  [N-BEATS HIỆU QUẢ CHO H=1?]
  → diag4 h=1: bars xanh lá (_tslope/_taccel/_detrend) trong top 10?
  → h=1 score ≥ 0.082 (rerun projection)?  target: ≥ v41+0.010

  [CS_LONG HIỆU QUẢ CHO H=25?]
  → diag4 h=25: bars tím (v_cs, bg_cs, l_cs) trong top 15?
  → h=25 score ≥ 0.2730 (v41/v38)?  target: tăng thêm
  → Nếu tím không xuất hiện top → CS_LONG không giúp → bỏ luôn

  [SO SÁNH 4 VARIANTS]
  v42a (main)  : baseline với 5 seeds
  v42b         : lambda_l2=15 h=25 → h=25 tăng? → regularize tốt hơn
  v42c         : num_leaves=127 h=10/25 → h=10 tăng? → deeper trees help?
  v42d         : feature_fraction=0.55 h=25 → randomness thêm giúp?
  → Cái nào có h=25 cao nhất → dùng làm base cho v43
''')


>>> v42 variant=a | seeds=5 | VAL_THRESHOLD=3500
>>> Feature map:
    h=1: ~208 feats (exclude 3: ['v_CS', 'bg_CS', 'l_CS']...)
    h=3: ~194 feats (exclude 17: ['v_ema_cross', 'v_roll_std_20', 'v_lag5']...)
    h=10: ~193 feats (exclude 18: ['cg_ema_cross', 'v_ema_cross', 'v_roll_std_20']...)
    h=25: ~196 feats (exclude 15: ['cg_ema_cross', 'v_ema_cross', 'v_roll_std_20']...)
⏳ FE v42 (variant=a | N-BEATS h=1 | CS h=25 | 5 seeds)...
   concat: 6,784,521 rows | 95 cols
   [RAM after concat]: 15,833 MB (15.5 GB)
   ✓ CS_LONG features: feature_v_cs, feature_bg_cs, feature_l_cs
   ✓ v38 lag/rolling/DiffRoC: 5 source cols
   [v42] Computing N-BEATS decomposition features...
   ✓ N-BEATS features: 8 cols (will be filtered to h=1 only)
   Total FE cols: 220
   [RAM before to_pandas]: 23,380 MB (22.8 GB)
   [RAM FE done]: 29,154 MB (28.5 GB)
   [RAM after FE]: 29,154 MB (28.5 GB)

 🚀 HORIZON 1  [variant=a]
Data: train=1,394,653, test=379,617 | Feats=208 [surgical=7 nbeats=8 cs=0]
   -> See

In [2]:
import os
for f in ['submission.csv', f'oof_v42{VARIANT_ID}.csv',
          'diag1_scores.png', 'diag2_pred_vs_true.png',
          'diag3_temporal_residual.png', 'diag4_feature_importance.png']:
    kb = os.path.getsize(f)//1024 if os.path.exists(f) else 0
    print(f'  {f}: {kb} KB' if kb else f'  {f}: MISSING')
print(f'\n✅ submission.csv (variant={VARIANT_ID}) sẵn sàng nộp!')

  submission.csv: 84392 KB
  oof_v42a.csv: 13567 KB
  diag1_scores.png: 40 KB
  diag2_pred_vs_true.png: 86 KB
  diag3_temporal_residual.png: 195 KB
  diag4_feature_importance.png: 187 KB

✅ submission.csv (variant=a) sẵn sàng nộp!
